# 逻辑回归的Torch面向对象实现

In [3]:
from sklearn.datasets import load_iris
from torch.utils.data import Dataset, DataLoader  # 数据集的统一处理与分批处理
from torch import nn  # 面向对象的封装
from torch import optim  # 梯度更新的封装
import torch

## 1.数据集的统一处理模式

- 数据集的集成
    - 继承Dataset
    - override覆盖
        - 下标运算符__getitem__
            - 访问数据
        - 长度运算符__len__
            - 访问数据数量
    - 实现构造器
        - 初始化数据

In [5]:
class IrisDataset(Dataset):
    """数据集类
    
    """
    def __init__(self):
        super(IrisDataset, self).__init__()
        data, target = load_iris(return_X_y = True)
        self.x = torch.tensor(data[50:150], dtype=torch.float32)
        self.y = torch.tensor(target[0:100], dtype=torch.float32).view(-1, 1)

    def __getitem__(self, idx):
        """重载下标运算符
        params:
            idx: 访问数据的下标
        return:
            指定数据的样本与标签
        """
        return self.x[idx], self.y[idx]

    def __len__(self):
        """
        """
        return self.x.shape[0]
        

In [13]:
ds = IrisDataset()
loader = DataLoader(ds, batch_size=20, shuffle=True) # batch_size 批次, shuffle 打乱

In [14]:
for x, y in loader:
    print(x.shape)

torch.Size([20, 4])
torch.Size([20, 4])
torch.Size([20, 4])
torch.Size([20, 4])
torch.Size([20, 4])


## 2. 模型的统一实现模式

- 深度学习可以拆分为两部
    - 特征抽取
    - 分类
通常特征抽取要经过好多层，到最后一层进行分类

In [15]:
class ClassfyModel(nn.Module):
    """
    """
    def __init__(self, in_features=4, out_features=1):
        """
        params:
            in_features: 分类要处理的数据维度
            out_features: 分类神经元数量
        """
        super(ClassfyModel, self).__init()
        # 线性运算
        self.fc = nn.linear(
            in_features=in_features,
            out_features=out_features,
            bias=True,
            dtype=torch.float32, # 默认float32
            device="cuda:0"
        ) # fc 既是神经元，全连接 ?

        # 概率运算
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        """推理函数, 执行推理计算
        params:
            x: 特征样本
        return:
            处理后的预测概率结果
        """
        y_ = self.fc(x) # 本质是调用运算符__call__
        y_ = self.sigmoid(y_)
        return y_

    
    

In [16]:
model = ClassfyModel(4, 1)

for x, y in loader:
    x = x.cuda()
    y = y.cuda()
    y_ = model(x)
    break

AttributeError: 'super' object has no attribute '_ClassfyModel__init'

## 3. 训练过程的梯度更新实现模式 -- 优化器

In [19]:
from sklearn.datasets import load_iris
from torch.utils.data import Dataset, DataLoader   # 数据集的统一处理与分批次处理
from torch import nn     # 面向对象的封装
from torch import optim  # 梯度更新的封装
import torch

# ============================1. 数据集实现 =========================
class IrisDataset(Dataset):
    """
    类的文档注释
    """
    def __init__(self):
        #调用父类构造器
        super(IrisDataset, self).__init__()   # python ABC
        # 加载数据
        data, target = load_iris(return_X_y=True)
        # 转换为张量
        self.x = torch.tensor(data[50:150], dtype=torch.float32)
        self.y = torch.tensor(target[0:100], dtype=torch.float32).view(-1, 1)

    def __getitem__(self, idx):
        """
        重载[]运算符
        参数：
            idx：访问数据的下标
        返回：
            指定位置的数据，样本与标签
        """
        return self.x[idx], self.y[idx]

    def __len__(self):
        return self.x.shape[0]   # len(self.x)

# ===========================2. 模型实现 =============================
class ClassifyModel(nn.Module):
    # 1. 构造器
    def __init__(self, in_features=4, out_features=1):
        super(ClassifyModel, self).__init__()
        # 线性运算（使用类）
        self.fc = nn.Linear(   # fc = full connection  = y = xW + b
            in_features=in_features,
            out_features=out_features,
            bias=True,
            dtype=torch.float32, # 默认是float32，所以可以省略
            # device="cuda:0"
        )
        # 概率运算
        self.sigmoid = nn.Sigmoid()
    
    # 2. 推理
    def forward(self, x):  
        """
        执行推理计算
        参数：
            x: 输入特征样本
        返回：
            预测的结果，也是概率
        """
        y_ = self.fc(x)   # fc也称可调用对象，本质是调用运算符__call__ == forward
        y_ = self.sigmoid(y_)
        return y_

# ===========================3. 训练 =============================
# 外部参数  --  超参数
epoch = 10000
lr = 0.001

# dataset
ds = IrisDataset()
loader = DataLoader(ds, batch_size=20, shuffle=True)

# 模型
model = ClassifyModel(4, 1)

# 损失函数
loss_f = nn.BCELoss()

# 优化器
optimizer = optim.Adam(
    params=model.parameters(),  # 取出模型所有的参数， 也是优化器更新的目标
    lr=lr
)

# 循环训练
device = "cuda:0"
model.to(device) # 移动模型到设备
for e in range(epoch):
    for x, y in loader:
        
        # 移动数据到设备
        x = x.to(device)
        y = y.to(device)
        # 推理
        y_ = model(x)
        # 损失值
        loss = loss_f(y_, y)
        # 求导
        loss.backward()
        # 梯度更新
        optimizer.step()
        # 梯度值清零
        optimizer.zero_grad()
    if e % 1000 == 0:
        with torch.no_grad():
            num_accu=0
            num_sample=0
            for x, y in loader:
                num_sample += len(x)
                x = x.to(device)
                y = y.to(device)

                y_ = model(x)
                y_[ y_ > 0.5 ] = 1
                y_[ y_ <= 0.5 ] = 0
                num_accu += (y == y_).float().sum()
            rate = num_accu / num_sample
            print(f"轮次:{e:05d}, 损失值:{loss.cpu().item():.6f}, 准确率:{rate.cpu().item() * 100:.2f}")

轮次:00000, 损失值:0.723395, 准确率:50.00
轮次:01000, 损失值:0.327604, 准确率:96.00
轮次:02000, 损失值:0.249172, 准确率:97.00
轮次:03000, 损失值:0.164556, 准确率:97.00
轮次:04000, 损失值:0.098836, 准确率:97.00
轮次:05000, 损失值:0.155863, 准确率:97.00
轮次:06000, 损失值:0.113795, 准确率:97.00
轮次:07000, 损失值:0.096443, 准确率:97.00
轮次:08000, 损失值:0.058579, 准确率:97.00
轮次:09000, 损失值:0.086031, 准确率:97.00
